In [164]:
# Required imports
import gempy as gp
import gempy_viewer as gpv

# Additional import (for topography)
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from gempy_engine.config import AvailableBackends
from gempy_engine.core.data.stack_relation_type import StackRelationType

# interactive thing
import ipywidgets as widgets
from matplotlib.collections import LineCollection
import scipy.ndimage
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors

import os

In [165]:
# Set some variables for the entire notebook
resolution = [125, 50, 50]
extent = [0, 2500, 0, 1000, 0, 1000]
refinement = 7
div = np.array([1, 2, 4, 8, 16, 32])
margin = 10
cwd = os.getcwd()

## Interactive Octree and Gempy model plot

In [170]:
# Load files
octrees = np.load("data/octrees.npy", allow_pickle=True)
lith_blocks = np.load("data/lith_blocks.npy", allow_pickle=True)
fault_blocks = np.load("data/fault_blocks.npy", allow_pickle=True)
scalar_field_at_surface_points = np.load('data/scalar_field_levels_per_surfaces.npy', allow_pickle=True)
scalar_field_matrices = np.load('data/scalar_field_matrices.npy', allow_pickle=True) 

# load input files 
surface_points=pd.read_csv(cwd + "/data/model7_surface_points_df.csv")
orientations=pd.read_csv(cwd + "/data/model7_orientations_df.csv")

In [171]:
# Precompute nr of octree nodes
octree_nodes = np.zeros((octrees.shape), dtype=int)

for model in range(octrees.shape[0]):
    data = np.zeros((1,2))
    for level in range(octrees.shape[1]):
        rounded = np.round(octrees[model][level],2)
        data = np.vstack((data, np.delete(rounded, 1, axis=1)))
        unique_data = np.unique(data, axis=0)
        octree_nodes[model,level]= len(unique_data)

# Precomputations
precomputed_data_all = []

for i in range(octrees.shape[0]): # 4 is number of models
    # Precompute once for all levels (up to 6, can adjust for more levels)
    precomputed_data = {}
    
    for level in range(refinement-1):  
        # Get coordinates
        x = octrees[i][level][:, 0]
        y = octrees[i][level][:, 2]

        # Standard spacings for this level
        standard_spacing_x = ((extent[1]-extent[0])/2) / div[level]
        standard_spacing_y = ((extent[5]-extent[4])/2) / div[level]

        # Get unique x and y values with indices
        unique_x, indices_x = np.unique(np.round(x, 0), return_inverse=True)
        unique_y, indices_y = np.unique(np.round(y, 0), return_inverse=True)

        # Map x to corresponding y-values and vice-versa
        x_to_y = [y[indices_x == i] for i in range(len(unique_x))]
        y_to_x = [x[indices_y == i] for i in range(len(unique_y))]

        # Store the precomputed values in a dictionary
        precomputed_data[level] = {
            'unique_x': unique_x,
            'unique_y': unique_y,
            'x_to_y': x_to_y,
            'y_to_x': y_to_x,
            'standard_spacing_x': standard_spacing_x,
            'standard_spacing_y': standard_spacing_y
        }
    precomputed_data_all.append(precomputed_data)


In [214]:
def gempy_octree_fault_interactive(show_input=True, show_field=False, show_lines=False, show_block=False,
                                   model=1, grid_toggle='None', octree_level=1, regular_level=1):

    model = int(model)
    octree_level = int(octree_level)
    
    fig, ax = plt.subplots()
    # Set the x and y extent of the model
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[4], extent[5])
    ax.set_title("Geologic model cross section")

    cmap_colors = ['#4285f4', '#ea4335', '#fbbc05', '#34a853', '#673ab7',
          '#c4e4fc', '#ffd4d4', '#fff4c2', '#c4f8bd']

    # color fixing just here for the input data
    cmap_colors2 = ['#ea4335', '#fbbc05', '#34a853', '#000000']

    # Define unique formations
    formations = surface_points['formation'].unique()

    # Create a color map and norm
    cmap = mcolors.ListedColormap(cmap_colors2)
    norm = mcolors.BoundaryNorm(np.arange(len(formations) + 1) - 0.5, len(formations))

    if show_input==True:
        # Plot the surface points
        sc = ax.scatter(surface_points.X, surface_points.Z,
                        c=surface_points['formation'].apply(lambda x: np.where(formations == x)[0][0]),
                        cmap=cmap,
                        norm=norm,
                        edgecolors="black")
    
        # Plot the orientations, removed for now for simplicity
        # ax.quiver(orientations.X, orientations.Z,
        #               orientations.G_x, orientations.G_z,
        #               orientations['formation'].apply(lambda x: np.where(formations == x)[0][0]),
        #               angles='xy', scale_units='xy',
        #               cmap=cmap,
        #               norm=norm,
        #               edgecolors="black",
        #               linewidth=1)
    
    smoothed_lith = scipy.ndimage.zoom(lith_blocks[model].reshape(resolution)[:,25,:].T, 3)
    smoothed_fault = scipy.ndimage.zoom(fault_blocks[model].reshape(resolution)[:,25,:].T, 3)

    # This one I have to check where to get from
    faults = [0]  #list(self.model.faults.df[self.model.faults.df['isFault'] == True].index)
    # scalar_field_matrix = scalar_field_matrices[model].reshape(3, *resolution)[:,:,25,:]
    scalar_field_matrix = scalar_field_matrices[model].reshape(scalar_field_matrices[model].shape[0], *resolution)[:,:,25,:]
    counter = scalar_field_matrices[model].shape[0] # nr of structural groups
    zorder=2

    if show_field==True: 
            # Combined scalar field just hacked for this model specifically
            combined_scalar_field = np.where(lith_block <= 2, scalar_field_matrix[1].T+0.04, scalar_field_matrix[2].T)
                
            ax.contourf(combined_scalar_field, 0, levels=15, cmap="inferno",
                                    linestyles='solid', origin='lower', alpha=0.5,
                                    zorder=zorder-52, extent=[extent[0], extent[1], extent[4], extent[5]])
        
    counters = np.arange(0, counter, 1)
    c_id = 0  # color id startpoint
    for f_id in counters:
        block = scalar_field_matrix[f_id].T
        level = scalar_field_at_surface_points[model][f_id][np.where(scalar_field_at_surface_points[model][f_id] != 0)]
        
        levels = np.insert(level, 0, block.max())
        c_id2 = c_id + len(level)
        if f_id == counters.max():
            levels = np.insert(levels, level.shape[0], block.min())
            c_id2 = c_id + len(levels)  # color id endpoint
        
        zorder = zorder - (f_id + len(level))
        if f_id not in faults:  #wenn es eine Schicht ist
            color = cmap_colors[c_id:c_id2]
            if len(color)>1:
                color=color[:-1][::-1] #letzte Farbe wegwerfen und Reihenfolge umdrehen
            if show_block==True:
                ax.contourf(block, 0, levels=np.sort(levels), colors=color,
                                        linestyles='solid', origin='lower', alpha=1,
                                        zorder=zorder-50, extent=[extent[0], extent[1], extent[4], extent[5]])
            # if show_field==True:
            #     ax.contourf(block, 0, levels=15, cmap="inferno",
            #                             linestyles='solid', origin='lower', alpha=0.2,
            #                             zorder=zorder-52, extent=[extent[0], extent[1], extent[4], extent[5]])
            if show_lines==True:
                ax.contour(block, 0, levels=np.sort(levels), colors=color,
                                           linestyles='solid', origin='lower',
                                          zorder=zorder, extent=[extent[0], extent[1], extent[4], extent[5]])
        else: # wenn es eine Störung ist
            color = cmap_colors[c_id:c_id2][0]
            # if show_lines==True:
            # always show fault for now
            ax.contour(block, 0, levels=np.sort(levels), colors="black",
                                linestyles='solid', origin='lower',
                                zorder=zorder, extent=[extent[0], extent[1], extent[4], extent[5]])
                 
        c_id += len(level)


    # Plotting the octree grid
    if grid_toggle == 'Octree':
        text1 = ax.text(2525, 900, f"Octree grid nodes: {(octree_nodes[model,octree_level-1]):04d}", fontsize=12, color='black', ha='left', va='center')
        nr_grid_nodes = (len(np.arange(extent[4], extent[5], extent[5]/(2**regular_level)))+1)**2
        text2 = ax.text(2525, 800, f"Regular grid nodes: 0000", fontsize=12, color='black', ha='left', va='center')
        # print("Nr of grid nodes: ", octree_nodes[model,octree_level-1])
        # Store line segments for batch plotting
        vertical_lines = []
        horizontal_lines = []
    
        # Iterate through levels to show
        for level in range(octree_level):
            # Retrieve precomputed data
            data = precomputed_data_all[model][level]
            unique_x = data['unique_x']
            unique_y = data['unique_y']
            x_to_y = data['x_to_y']
            y_to_x = data['y_to_x']
            standard_spacing_x = data['standard_spacing_x']
            standard_spacing_y = data['standard_spacing_y']
            
    
            # Draw vertical lines only between existing points with standard spacing and margin
            for i, y_vals_at_x in enumerate(x_to_y):
                y_vals_at_x = np.sort(y_vals_at_x)
                if len(y_vals_at_x) > 1:
                    diffs = np.diff(y_vals_at_x)
                    valid_indices = np.where(np.abs(diffs - standard_spacing_y) <= margin)[0]
                    for idx in valid_indices:
                        vertical_lines.append(((unique_x[i], y_vals_at_x[idx]), (unique_x[i], y_vals_at_x[idx + 1])))
    
            # Draw horizontal lines only between existing points with standard spacing and margin
            for i, x_vals_at_y in enumerate(y_to_x):
                x_vals_at_y = np.sort(x_vals_at_y)
                if len(x_vals_at_y) > 1:
                    diffs = np.diff(x_vals_at_y)
                    valid_indices = np.where(np.abs(diffs - standard_spacing_x) <= margin)[0]
                    for idx in valid_indices:
                        horizontal_lines.append(((x_vals_at_y[idx], unique_y[i]), (x_vals_at_y[idx + 1], unique_y[i])))
    
        # Convert line segments to collections for faster plotting
        v_line_collection = LineCollection(vertical_lines, colors='black', zorder=-10, linewidths=1)
        h_line_collection = LineCollection(horizontal_lines, colors='black', zorder=-10, linewidths=1)
    
        # Add the line collections to the plot
        ax.add_collection(v_line_collection)
        ax.add_collection(h_line_collection)

    # Plotting the regular grid
    elif grid_toggle == 'Regular':
        text1 = ax.text(2525, 900, f"Octree grid nodes: 0000", fontsize=12, color='black', ha='left', va='center')
        nr_grid_nodes = (len(np.arange(extent[4], extent[5], extent[5]/(2**regular_level)))+1)**2
        text2 = ax.text(2525, 800, f"Regular grid nodes: {nr_grid_nodes:04d}", fontsize=12, color='black', ha='left', va='center')
        
        ax.hlines(np.arange(extent[4], extent[5], extent[5]/(2**regular_level)), 
                  xmin=extent[0], xmax=extent[1], color="black", linewidth=1, zorder=-10)
        ax.vlines(np.arange(extent[0], extent[1], extent[1]/(2**regular_level)), 
                  ymin=extent[0], ymax=extent[1], color="black", linewidth=1, zorder=-10)

    else:
        text1 = ax.text(2525, 900, f"Octree grid nodes: 0000", fontsize=12, color='black', ha='left', va='center')
        text2 = ax.text(2525, 800, f"Regular grid nodes: 0000", fontsize=12, color='black', ha='left', va='center')


    # Example lists of colors and corresponding labels
    # blue, red, yellow, green, purple
    label_colors = ['#000000', '#ea4335', '#fbbc05', '#34a853']
    labels = ["rock1", "rock2", "rock3", "fault"][::-1]
    
    # Create a list of Line2D objects for the legend
    legend_elements = [Line2D([0], [0], color=color, lw=4) for color in label_colors]
    
    # Add the legend to the figure with the provided labels and colors
    lgd = ax.legend(legend_elements, labels, loc='center left', bbox_to_anchor=(1, 0.5), title="Legend")
    
    # Grid lines and labels
    ax.set_xlabel('X')
    ax.set_ylabel('Z')
    ax.set_aspect('equal')

    # if grid_toggle == 'Octree':
    #     plt.savefig("images\\Plot_Model{}_C_OctreeGrid_Level{}.png".format(model, octree_level), 
    #                                                                         dpi=150, bbox_extra_artists=(lgd, text1), bbox_inches='tight')
    # elif grid_toggle == 'Regular':
    #     plt.savefig("images\\Plot_Model{}_B_RegularGrid_Level{}.png".format(model, regular_level), 
    #                                                                         dpi=150, bbox_extra_artists=(lgd, text1), bbox_inches='tight')
    # else:
    #     plt.savefig("images\\Plot_Model{}_A_NoGrid_Level{}.png".format(model, "_None"), 
    #                                                                         dpi=150, bbox_extra_artists=(lgd, text1), bbox_inches='tight')

    plt.show()

In [215]:
# Create sliders for frequency and amplitude
input_toggle = widgets.ToggleButton(value=True, description='Show input data', disabled=False, button_style='')
field_toggle = widgets.ToggleButton(value=False, description='Show field solution', disabled=False, button_style='')
lines_toggle = widgets.ToggleButton(value=False, description='Show contour solution', disabled=False, button_style='')
block_toggle = widgets.ToggleButton(value=False, description='Show block solution', disabled=False, button_style='')

model_slider = widgets.IntSlider(value=0, min=0, max=3, step=1, description='Move fault location', layout=widgets.Layout(width='400px'))

grid_toggle = widgets.ToggleButtons(options=['None', 'Regular', 'Octree'],description='Show grid:',disabled=False,button_style='')

regular_slider = widgets.IntSlider(value=1, min=1, max=6, step=1, description='Regular level', layout=widgets.Layout(width='400px'))
octree_slider = widgets.IntSlider(value=1, min=1, max=6, step=1, description='Octree level', layout=widgets.Layout(width='400px'))


model_slider.style = {'description_width': '120px'}
octree_slider.style = {'description_width': '120px'}
regular_slider.style = {'description_width': '120px'}

model_slider.style.handle_color = 'black'
octree_slider.style.handle_color = 'grey'
regular_slider.style.handle_color = 'grey'

# Use interact to link sliders to the plotting function
widgets.interact(gempy_octree_fault_interactive, 
                 show_input=input_toggle,
                 show_field=field_toggle, show_lines=lines_toggle, show_block=block_toggle,
                 model=model_slider, 
                 grid_toggle=grid_toggle,
                 octree_level=octree_slider, regular_level=regular_slider);

interactive(children=(ToggleButton(value=True, description='Show input data'), ToggleButton(value=False, descr…